# T04 — Boundary / particle-validity policy manual check

Scope (per `tasks.md` T04 and the 2026-09-22 task instructions):

- `iste.boundary.ParticleState` — minimal vectorized particle position + validity flag;
- `iste.boundary.apply_boundary_policy` — the `README.md` stop-on-invalid policy: a
  proposed step into an invalid/land cell, or out of the valid domain, marks the
  particle invalid and stops its propagation (no reflection/absorption/clipping/
  wrapping/re-entry); once invalid, a particle stays invalid;
- `iste.boundary.netcdf_validity_check` — an adapter reusing T02's existing
  `interpolate_field` validity semantics, so "valid" is never redefined here.

No Euler-Maruyama propagation, particle weights, `grad(K)`/`div(U)` usage, forcing
ensemble, observation likelihood, KDE, SMC, MCMC, or AIS is implemented or exercised
here. This notebook uses a single tiny synthetic domain and, at the end, one real
ocean-grid example -- no 7-day simulation, no large particle counts.

In [1]:
import os
import sys

REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), os.pardir))
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

import numpy as np

from iste.boundary import ParticleState, apply_boundary_policy, netcdf_validity_check
from iste.environment import load_field_dataset

OCEAN_PATH = os.path.join(REPO_ROOT, "data", "ocean.nc")


def synthetic_domain_validity(lat, lon):
    """Tiny hand-defined domain: ocean is [0,10] x [0,10], with a 'land' square
    [4,6] x [4,6] carved out of the middle."""
    lat = np.asarray(lat, dtype=np.float64)
    lon = np.asarray(lon, dtype=np.float64)
    in_domain = (lat >= 0.0) & (lat <= 10.0) & (lon >= 0.0) & (lon <= 10.0)
    is_land = (lat >= 4.0) & (lat <= 6.0) & (lon >= 4.0) & (lon <= 6.0)
    return in_domain & ~is_land


def show(label, before, after):
    print(f"{label}:")
    print(f"  before: lat={before.lat[0]}, lon={before.lon[0]}, valid={before.valid[0]}")
    print(f"  after:  lat={after.lat[0]}, lon={after.lon[0]}, valid={after.valid[0]}")

## Case 1: one valid particle, proposed step stays in open water

In [2]:
before1 = ParticleState.initialize(lat0=[1.0], lon0=[1.0])
after1 = apply_boundary_policy(before1, [2.0], [2.0], synthetic_domain_validity)
show("valid particle, valid proposed step", before1, after1)
assert after1.valid[0] and after1.lat[0] == 2.0

valid particle, valid proposed step:
  before: lat=1.0, lon=1.0, valid=True
  after:  lat=2.0, lon=2.0, valid=True


## Case 2: one particle hitting land

In [3]:
before2 = ParticleState.initialize(lat0=[3.0], lon0=[3.0])
after2 = apply_boundary_policy(before2, [5.0], [5.0], synthetic_domain_validity)  # (5,5) is land
show("particle proposing a step into land", before2, after2)
assert not after2.valid[0]
assert after2.lat[0] == 3.0 and after2.lon[0] == 3.0, "must stay at its last VALID position, not move to land"

particle proposing a step into land:
  before: lat=3.0, lon=3.0, valid=True
  after:  lat=3.0, lon=3.0, valid=False


## Case 3: one particle leaving the spatial domain

In [4]:
before3 = ParticleState.initialize(lat0=[9.0], lon0=[9.0])
after3 = apply_boundary_policy(before3, [11.0], [11.0], synthetic_domain_validity)  # outside [0,10]
show("particle proposing a step out of the domain", before3, after3)
assert not after3.valid[0]
assert after3.lat[0] == 9.0 and after3.lon[0] == 9.0

particle proposing a step out of the domain:
  before: lat=9.0, lon=9.0, valid=True
  after:  lat=9.0, lon=9.0, valid=False


## Case 4: an already-invalid particle attempted again

Even though the next proposed step (`(1, 1)`) is a perfectly valid open-water location,
the particle must remain invalid and unmoved -- "once invalid, a particle stays
invalid".

In [5]:
before4 = after2  # already invalid, from Case 2
after4 = apply_boundary_policy(before4, [1.0], [1.0], synthetic_domain_validity)
show("already-invalid particle, valid-looking next proposal", before4, after4)
assert not after4.valid[0]
assert after4.lat[0] == 3.0 and after4.lon[0] == 3.0, "must not move even though the new proposal was valid"

already-invalid particle, valid-looking next proposal:
  before: lat=3.0, lon=3.0, valid=False
  after:  lat=3.0, lon=3.0, valid=False


## All four cases together: invalid particles are retained, not dropped

In [6]:
combined = ParticleState(
    lat=np.array([1.0, 3.0, 9.0]),
    lon=np.array([1.0, 3.0, 9.0]),
    valid=np.array([True, True, True]),
)
combined_after = apply_boundary_policy(combined, [2.0, 5.0, 11.0], [2.0, 5.0, 11.0], synthetic_domain_validity)
print("lat:  ", combined_after.lat)
print("lon:  ", combined_after.lon)
print("valid:", combined_after.valid)
print(f"n_particles={combined_after.n_particles} (unchanged), "
      f"valid_count={combined_after.valid_count}, invalid_count={combined_after.invalid_count}, "
      f"valid_particle_fraction={combined_after.valid_particle_fraction:.3f}")
assert combined_after.n_particles == 3

lat:   [2. 3. 9.]
lon:   [2. 3. 9.]
valid: [ True False False]
n_particles=3 (unchanged), valid_count=1, invalid_count=2, valid_particle_fraction=0.333


## Real data: reusing T02's validity semantics on the ocean grid

`netcdf_validity_check` does not redefine "valid" -- it is a thin adapter over
`interpolate_field`. Here we take one real masked/land cell from `data/ocean.nc` and
show a particle proposing a step onto it becomes invalid, while a step to a nearby
valid cell succeeds.

In [7]:
ocean = load_field_dataset(OCEAN_PATH, ["uo", "vo"], time_dim="time", lat_dim="latitude", lon_dim="longitude")
t0 = ocean.time[0]
check = netcdf_validity_check(ocean, "uo", t0)

uo0 = ocean.data["uo"][0]
land_i, land_j = np.argwhere(np.isnan(uo0))[0]
land_lat, land_lon = float(ocean.lat[land_i]), float(ocean.lon[land_j])

# start the particle somewhere safely inside the domain (not the land cell itself).
start_lat = float(ocean.lat[len(ocean.lat) // 2])
start_lon = float(ocean.lon[len(ocean.lon) // 2])
before5 = ParticleState.initialize(lat0=[start_lat], lon0=[start_lon])
after5 = apply_boundary_policy(before5, [land_lat], [land_lon], check)
show("real ocean grid: proposing a step onto a masked/land cell", before5, after5)
assert not after5.valid[0]

c:\Users\venka\AppData\Local\Programs\Python\Python313\Lib\site-packages\argopy\utils\lists.py:38: UserWarning: An error occurred while loading the ERDDAP data fetcher, it will not be available !
<class 'ImportError'>
cannot import name '_quote_string_constraints' from 'erddapy.erddapy' (c:\Users\venka\AppData\Local\Programs\Python\Python313\Lib\site-packages\erddapy\erddapy.py)
  warnings.warn(
c:\Users\venka\AppData\Local\Programs\Python\Python313\Lib\site-packages\argopy\utils\lists.py:50: UserWarning: An error occurred while loading the ArgoVis data fetcher, it will not be available !
<class 'ImportError'>
cannot import name '_quote_string_constraints' from 'erddapy.erddapy' (c:\Users\venka\AppData\Local\Programs\Python\Python313\Lib\site-packages\erddapy\erddapy.py)
  warnings.warn(
c:\Users\venka\AppData\Local\Programs\Python\Python313\Lib\site-packages\argopy\utils\lists.py:66: UserWarning: An error occurred while loading the GDAC data fetcher, it will not be available !
<class 

real ocean grid: proposing a step onto a masked/land cell:
  before: lat=33.27083206176758, lon=31.625, valid=True
  after:  lat=33.27083206176758, lon=31.625, valid=False


## Summary / known limitations

- All four required before/after cases behave exactly per `README.md`'s stop-on-invalid
  policy: valid step -> move and stay valid; land or out-of-domain step -> mark invalid,
  do not move, retain the invalid status; an already-invalid particle never moves again
  regardless of what is proposed next.
- Invalid particles are always retained in `ParticleState`'s arrays (`n_particles` never
  shrinks), and `valid_count`/`invalid_count`/`valid_particle_fraction` are available for
  future diagnostics (T10) to build on.
- "Valid" is defined exactly as T02's `interpolate_field` already defines it -- this
  module adds no new domain/land model of its own.
- No reflection, absorption, clipping, wrapping, or re-entry rule exists -- only the
  explicitly required stop-on-invalid policy.
- No Euler-Maruyama propagation, particle weights, `grad(K)`/`div(U)` usage, forcing
  ensemble, observation likelihood, KDE, SMC, MCMC, or AIS exists in this repository --
  `apply_boundary_policy` only decides what happens to a single *already-proposed* step;
  generating that proposed step (via the backward SDE) is T05.